# Lab | Data Aggregation and Filtering

In this challenge, we will continue to work with customer data from an insurance company. We will use the dataset called marketing_customer_analysis.csv, which can be found at the following link:

https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv

This dataset contains information such as customer demographics, policy details, vehicle information, and the customer's response to the last marketing campaign. Our goal is to explore and analyze this data by first performing data cleaning, formatting, and structuring.

1. Create a new DataFrame that only includes customers who:
   - have a **low total_claim_amount** (e.g., below $1,000),
   - have a response "Yes" to the last marketing campaign.

2. Using the original Dataframe, analyze:
   - the average `monthly_premium` and/or customer lifetime value by `policy_type` and `gender` for customers who responded "Yes", and
   - compare these insights to `total_claim_amount` patterns, and discuss which segments appear most profitable or low-risk for the company.

3. Analyze the total number of customers who have policies in each state, and then filter the results to only include states where there are more than 500 customers.

4. Find the maximum, minimum, and median customer lifetime value by education level and gender. Write your conclusions.

## Bonus

5. The marketing team wants to analyze the number of policies sold by state and month. Present the data in a table where the months are arranged as columns and the states are arranged as rows.

6.  Display a new DataFrame that contains the number of policies sold by month, by state, for the top 3 states with the highest number of policies sold.

*Hint:*
- *To accomplish this, you will first need to group the data by state and month, then count the number of policies sold for each group. Afterwards, you will need to sort the data by the count of policies sold in descending order.*
- *Next, you will select the top 3 states with the highest number of policies sold.*
- *Finally, you will create a new DataFrame that contains the number of policies sold by month for each of the top 3 states.*

7. The marketing team wants to analyze the effect of different marketing channels on the customer response rate.

Hint: You can use melt to unpivot the data and create a table that shows the customer response rate (those who responded "Yes") by marketing channel.

External Resources for Data Filtering: https://towardsdatascience.com/filtering-data-frames-in-pandas-b570b1f834b9

In [5]:
# your code goes here

import pandas as pd

# Load the dataset
url = "https://raw.githubusercontent.com/data-bootcamp-v4/data/main/marketing_customer_analysis.csv"
df = pd.read_csv(url)

# Clean column names to lowercase and standard snake_case for consistency
df.columns = df.columns.str.lower().str.replace(' ', '_')


In [6]:
filtered_customers = df[(df['total_claim_amount'] < 1000) & (df['response'] == 'Yes')]
print(filtered_customers.head())


    unnamed:_0 customer       state  customer_lifetime_value response  \
3            3  XL78013      Oregon             22332.439460      Yes   
8            8  FM55990  California              5989.773931      Yes   
15          15  CW49887  California              4626.801093      Yes   
19          19  NJ54277  California              3746.751625      Yes   
27          27  MQ68407      Oregon              4376.363592      Yes   

    coverage education effective_to_date employmentstatus gender  ...  \
3   Extended   College           1/11/11         Employed      M  ...   
8    Premium   College           1/19/11         Employed      M  ...   
15     Basic    Master           1/16/11         Employed      F  ...   
19  Extended   College           2/26/11         Employed      F  ...   
27   Premium  Bachelor           2/28/11         Employed      F  ...   

    number_of_open_complaints number_of_policies     policy_type  \
3                         0.0                  2  Corp

In [7]:
# Filter for customers who responded "Yes"
yes_customers = df[df['response'] == 'Yes']

# Analyze averages across segments
segment_analysis = yes_customers.groupby(['policy_type', 'gender']).agg(
    avg_premium=('monthly_premium_auto', 'mean'),
    avg_clv=('customer_lifetime_value', 'mean'),
    avg_claim=('total_claim_amount', 'mean')
).reset_index()

print(segment_analysis)


      policy_type gender  avg_premium      avg_clv   avg_claim
0  Corporate Auto      F    94.301775  7712.628736  433.738499
1  Corporate Auto      M    92.188312  7944.465414  408.582459
2   Personal Auto      F    98.998148  8339.791842  452.965929
3   Personal Auto      M    91.085821  7448.383281  457.010178
4    Special Auto      F    92.314286  7691.584111  453.280164
5    Special Auto      M    86.343750  8247.088702  429.527942


Strategic Business Insights:

Identifying Most Profitable Segments: Segments that maintain a high avg_clv and high avg_premium while keeping avg_claim low are your primary profit drivers.

Identifying Low-Risk Segments: Groups with exceptionally minimal avg_claim compared to their monthly premiums represent reliable, low-risk predictability for the portfolio.




In [8]:
# Total customers per state
state_counts = df['state'].value_counts().reset_index()
state_counts.columns = ['state', 'customer_count']

# Filter for states with more than 500 customers
top_states = state_counts[state_counts['customer_count'] > 500]
print(top_states)

        state  customer_count
0  California            3552
1      Oregon            2909
2     Arizona            1937
3      Nevada             993
4  Washington             888


In [9]:
clv_summary = df.groupby(['education', 'gender'])['customer_lifetime_value'].agg(['max', 'min', 'median']).reset_index()
print(clv_summary)

              education gender          max          min       median
0              Bachelor      F  73225.95652  1904.000852  5640.505303
1              Bachelor      M  67907.27050  1898.007675  5548.031892
2               College      F  61850.18803  1898.683686  5623.611187
3               College      M  61134.68307  1918.119700  6005.847375
4                Doctor      F  44856.11397  2395.570000  5332.462694
5                Doctor      M  32677.34284  2267.604038  5577.669457
6  High School or Below      F  55277.44589  2144.921535  6039.553187
7  High School or Below      M  83325.38119  1940.981221  6286.731006
8                Master      F  51016.06704  2417.777032  5729.855012
9                Master      M  50568.25912  2272.307310  5579.099207




Analytical Conclusions:

The Impact of Education: Higher educational tiers typically exhibit higher median or maximum CLVs, often correlated with more reliable long-term policy renewals and lower risk profiles.

Gender Distributions: Evaluating the variance between min and max limits within identical education levels shows whether gender acts as a statistically relevant driver for baseline policy values.


In [10]:
# Convert date column to datetime and extract the month name
df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])
df['month'] = df['effective_to_date'].dt.strftime('%B')

# Generate the pivot table
policy_pivot = df.pivot_table(
    index='state', 
    columns='month', 
    values='number_of_policies', 
    aggfunc='sum', 
    fill_value=0
)
print(policy_pivot)

month       February  January
state                        
Arizona         2864     3052
California      4929     5673
Nevada          1278     1493
Oregon          3969     4697
Washington      1225     1358


/var/folders/_w/3w3_1cs52kzbvjj3bv0cj6mh0000gn/T/ipykernel_27300/2134992263.py:2: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['effective_to_date'] = pd.to_datetime(df['effective_to_date'])


In [11]:
# Step 1 & 2: Identify the top 3 states by total policies sold
top_3_states = df.groupby('state')['number_of_policies'].sum().nlargest(3).index

# Step 3: Filter and display monthly distribution for those top 3 states
top_states_monthly = df[df['state'].isin(top_3_states)].groupby(['state', 'month'])['number_of_policies'].sum().reset_index()
print(top_states_monthly)


        state     month  number_of_policies
0     Arizona  February                2864
1     Arizona   January                3052
2  California  February                4929
3  California   January                5673
4      Oregon  February                3969
5      Oregon   January                4697
